[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semantica-agi/semantica/blob/main/cookbook/introduction/11_Chunking_and_Splitting.ipynb)

# 分块与切分 —— 综合指南

## 概述

本 Notebook 对 Semantica 的 split 模块进行了**全面演练**,展示了所有分块策略和方法,以实现最优的文档处理。你将学会使用 15 种以上的切分方法,包括标准方法、语义方法和感知知识图谱的方法。

**文档**:[API 参考](https://semantica.readthedocs.io/reference/split/)

### 学习目标

学完本 Notebook 后,你将能够:

- 使用 `TextSplitter` 调用多种方法
- 应用标准切分方法(recursive、token、sentence、paragraph)
- 使用语义分块以保持主题连贯
- 应用感知知识图谱的分块(entity-aware、relation-aware、graph-based)
- 使用专用分块器(structural、sliding window、table、hierarchical)
- 使用 `ProvenanceTracker` 追踪溯源
- 为你的使用场景选择合适的方法

### 你将学到什么

| 组件 | 用途 | 何时使用 |
|-----------|---------|-------------|
| `TextSplitter` | 统一的切分器 | 所有分块需求 |
| `SemanticChunker` | 语义边界 | 基于主题的分块 |
| `EntityAwareChunker` | 保留实体 | GraphRAG 工作流 |
| `RelationAwareChunker` | 保留三元组 | 知识图谱构建 |
| `StructuralChunker` | 文档结构 | 有格式的文档 |
| `HierarchicalChunker` | 多层级分块 | 大型文档 |

---

## 安装

从 PyPI 安装 Semantica:

```bash
pip install semantica
# 或安装全部可选依赖:
pip install semantica[all]
```

---

In [ ]:
!pip install -q semantica


## 步骤 1:使用 TextSplitter 进行基础分块

让我们从统一的 `TextSplitter` 接口开始,它提供了对所有分块方法的访问。

### 什么是 TextSplitter?

`TextSplitter` 是一个统一接口,支持 15 种以上的分块方法:
- **标准**:recursive、token、sentence、paragraph、character、word
- **语义**:semantic_transformer、llm、huggingface、nltk
- **知识图谱/本体**:entity_aware、relation_aware、graph_based、ontology_aware
- **高级**:hierarchical、structural、sliding_window、table

In [ ]:
from semantica.split import TextSplitter

# Sample long text
text = """
Apple Inc. is a technology company founded by Steve Jobs, Steve Wozniak, and Ronald Wayne 
in Cupertino, California on April 1, 1976. The company's current CEO is Tim Cook, who took 
over from Steve Jobs in August 2011. Apple is headquartered at One Apple Park Way in Cupertino.

Apple develops and sells consumer electronics, computer software, and online services. The company's 
hardware products include the iPhone smartphone, the iPad tablet computer, the Mac personal computer, 
the iPod portable media player, the Apple Watch smartwatch, the Apple TV digital media player, and the 
HomePod smart speaker.

Apple's software includes the macOS and iOS operating systems, the iTunes media player, the Safari web 
browser, and the iLife and iWork creativity and productivity suites. Its online services include the 
iTunes Store, the iOS App Store and Mac App Store, Apple Music, and iCloud.
"""

# Basic recursive splitting
splitter = TextSplitter(
    method="recursive",
    chunk_size=200,
    chunk_overlap=50
)

chunks = splitter.split(text)

print(f"Split into {len(chunks)} chunks using recursive method\n")
print("=" * 80)

for i, chunk in enumerate(chunks, 1):
    print(f"\nChunk {i}:")
    print(f"  Length: {len(chunk.text)} characters")
    print(f"  Start: {chunk.start_index}, End: {chunk.end_index}")
    print(f"  Text: {chunk.text[:100]}...")

print("\n" + "=" * 80)

## 步骤 2:标准切分方法

让我们比较不同的标准切分方法。

### 方法比较

| 方法 | 最适合 | 速度 | 准确度 |
|--------|----------|-------|----------|
| **recursive** | 通用文本 | 快 | 良好 |
| **sentence** | 连贯的分块 | 中等 | 很好 |
| **token** | LLM 上下文 | 中等 | 极佳 |
| **paragraph** | 自然断点 | 快 | 良好 |

In [ ]:
# Compare different methods
methods = ["recursive", "sentence", "paragraph"]

print("Comparing Standard Splitting Methods:\n")
print("=" * 80)

for method in methods:
    splitter = TextSplitter(
        method=method,
        chunk_size=200,
        chunk_overlap=50
    )
    
    chunks = splitter.split(text)
    
    print(f"\nMethod: {method.upper()}")
    print("-" * 40)
    print(f"  Chunks created: {len(chunks)}")
    print(f"  Avg chunk size: {sum(len(c.text) for c in chunks) / len(chunks):.0f} chars")
    print(f"  First chunk: {chunks[0].text[:80]}...")

print("\n" + "=" * 80)

## 步骤 3:基于 Token 的切分

在需要遵守 Token 上限的 LLM 应用中,基于 Token 的切分至关重要。

### 为什么基于 Token?

- **LLM 上下文窗口**:GPT-4 有 8K/32K 的 Token 上限
- **精确计数**:字符数 ≠ Token 数
- **成本优化**:Token 决定 API 成本

In [ ]:
from semantica.split import split_by_tokens

# Token-based splitting
chunks = split_by_tokens(
    text,
    chunk_size=100,  # 100 tokens
    chunk_overlap=20,
    tokenizer="tiktoken",
    model="gpt-4"
)

print("Token-Based Splitting Results:\n")
print("=" * 80)

for i, chunk in enumerate(chunks, 1):
    token_count = chunk.metadata.get('token_count', 'N/A')
    print(f"\nChunk {i}:")
    print(f"  Tokens: {token_count}")
    print(f"  Characters: {len(chunk.text)}")
    print(f"  Ratio: {len(chunk.text)/token_count if token_count != 'N/A' else 'N/A':.2f} chars/token")

print("\n" + "=" * 80)

## 步骤 4:语义分块

语义分块基于嵌入,按语义边界创建分块。

### 工作原理

1. 将文本切分为句子
2. 为每个句子生成嵌入
3. 计算相邻句子之间的相似度
4. 在相似度低于阈值的位置创建边界

In [ ]:
from semantica.split import SemanticChunker

# Semantic chunking
semantic_chunker = SemanticChunker(
    chunk_size=200,
    chunk_overlap=50,
    embedding_model="all-MiniLM-L6-v2",
    similarity_threshold=0.7
)

chunks = semantic_chunker.chunk(text)

print("Semantic Chunking Results:\n")
print("=" * 80)

for i, chunk in enumerate(chunks, 1):
    coherence = chunk.metadata.get('coherence_score', 'N/A')
    print(f"\nChunk {i}:")
    print(f"  Length: {len(chunk.text)} chars")
    print(f"  Coherence: {coherence}")
    print(f"  Text: {chunk.text[:100]}...")

print("\n" + "=" * 80)

## 步骤 5:面向 GraphRAG 的实体感知分块

实体感知分块会保留实体边界,这对 GraphRAG 工作流至关重要。

### 为什么要实体感知?

- **保留实体**:不要把 "Steve Jobs" 拆到不同分块中
- **更好的抽取**:完整的实体能提升 NER 准确率
- **GraphRAG**:对知识图谱构建不可或缺

In [ ]:
from semantica.split import EntityAwareChunker

# Entity-aware chunking
entity_chunker = EntityAwareChunker(
    chunk_size=200,
    chunk_overlap=50,
    ner_method="ml",  # "ml" (spaCy), "pattern", or "llm"
    preserve_entities=True
)

chunks = entity_chunker.chunk(text)

print("Entity-Aware Chunking Results:\n")
print("=" * 80)

for i, chunk in enumerate(chunks, 1):
    entities = chunk.metadata.get('entities', [])
    print(f"\nChunk {i}:")
    print(f"  Length: {len(chunk.text)} chars")
    print(f"  Entities: {len(entities)}")
    
    if entities:
        # Handle both Entity objects and dicts
        entity_texts = [e.get('text', e.get('entity', '')) if isinstance(e, dict) else str(e) for e in entities[:3]]
        print(f"  Sample entities: {entity_texts}")

print("\n" + "=" * 80)

## 步骤 6:关系感知分块

关系感知分块会把关系三元组保留在同一分块内。

### 为什么要关系感知?

- **保留三元组**:将 (主语, 谓语, 宾语) 放在一起
- **知识图谱构建**:更适合构建知识图谱
- **上下文**:关系需要完整的上下文

In [ ]:
from semantica.split import RelationAwareChunker

# Relation-aware chunking
relation_chunker = RelationAwareChunker(
    chunk_size=200,
    chunk_overlap=50,
    preserve_triplets=True
)

chunks = relation_chunker.chunk(text)

print("Relation-Aware Chunking Results:\n")
print("=" * 80)

for i, chunk in enumerate(chunks, 1):
    triplets = chunk.metadata.get('triplets', [])
    relationships = chunk.metadata.get('relationships', [])
    
    print(f"\nChunk {i}:")
    print(f"  Length: {len(chunk.text)} chars")
    print(f"  Triplets: {len(triplets)}")
    print(f"  Relationships: {len(relationships)}")

print("\n" + "=" * 80)

## 步骤 7:结构化分块

结构化分块会遵循标题、段落和列表等文档结构。

### 何时使用?

- **有格式的文档**:Markdown、HTML、结构化文本
- **保留层级**:让各章节保持在一起
- **更好的上下文**:标题提供上下文

In [ ]:
from semantica.split import StructuralChunker

# Markdown text with structure
markdown_text = """
# Apple Inc.

## History

Apple Inc. was founded by Steve Jobs, Steve Wozniak, and Ronald Wayne in 1976.

## Products

### Hardware
- iPhone
- iPad
- Mac

### Software
- macOS
- iOS
- Safari
"""

# Structural chunking
structural_chunker = StructuralChunker(
    respect_headings=True,
    respect_paragraphs=True,
    respect_lists=True,
    max_chunk_size=500
)

chunks = structural_chunker.chunk(markdown_text)

print("Structural Chunking Results:\n")
print("=" * 80)

for i, chunk in enumerate(chunks, 1):
    section = chunk.metadata.get('section_title', 'N/A')
    level = chunk.metadata.get('heading_level', 'N/A')
    
    print(f"\nChunk {i}:")
    print(f"  Section: {section}")
    print(f"  Level: {level}")
    print(f"  Text: {chunk.text[:80]}...")

print("\n" + "=" * 80)

## 步骤 8:层级分块

层级分块为大型文档创建多层级的分块。

### 好处

- **多种粒度**:文档 → 章节 → 段落
- **更好的导航**:父子关系
- **灵活的检索**:可在不同层级进行查询

In [ ]:
from semantica.split import HierarchicalChunker

# Hierarchical chunking
hierarchical_chunker = HierarchicalChunker(
    chunk_sizes=[400, 200, 100],  # 3 levels
    chunk_overlaps=[80, 40, 20],
    create_parent_chunks=True
)

chunks = hierarchical_chunker.chunk(text)

print("Hierarchical Chunking Results:\n")
print("=" * 80)

for i, chunk in enumerate(chunks, 1):
    level = chunk.metadata.get('level', 'N/A')
    parent_id = chunk.metadata.get('parent_id', None)
    child_ids = chunk.metadata.get('child_ids', [])
    
    print(f"\nChunk {i}:")
    print(f"  Level: {level}")
    print(f"  Length: {len(chunk.text)} chars")
    print(f"  Parent: {parent_id if parent_id else 'None (root)'}")
    print(f"  Children: {len(child_ids)}")

print("\n" + "=" * 80)

## 步骤 9:滑动窗口分块

滑动窗口会创建有重叠的固定大小分块。

### 使用场景

- **稠密检索**:确保不遗漏任何信息
- **固定上下文**:分块大小一致
- **重叠控制**:精确管理重叠

In [ ]:
from semantica.split import SlidingWindowChunker

# Sliding window chunking
sliding_chunker = SlidingWindowChunker(
    chunk_size=150,
    overlap=50
)

chunks = sliding_chunker.chunk(text)

print("Sliding Window Chunking Results:\n")
print("=" * 80)

for i, chunk in enumerate(chunks, 1):
    # Calculate overlap manually
    overlap = 0
    if i > 1:
        prev_chunk = chunks[i-2]
        overlap = max(0, prev_chunk.end_index - chunk.start_index)
    
    print(f"\nWindow {i}:")
    print(f"  Position: {chunk.start_index}-{chunk.end_index}")
    print(f"  Length: {len(chunk.text)} chars")
    print(f"  Overlap with previous: {overlap} chars")

print("\n" + "=" * 80)

## 步骤 10:表格分块

表格分块在切分大表格的同时保留表格结构。

### 特性

- **保留表头**:在每个分块中保留列标题
- **按行切分**:按行而不是按字符切分
- **包含上下文**:纳入周围的文本

In [ ]:
from semantica.split import TableChunker

# Text with table
text_with_table = """
Apple's product lineup includes:

| Product | Category | Release Year |
|---------|----------|-------------|
| iPhone | Smartphone | 2007 |
| iPad | Tablet | 2010 |
| Mac | Computer | 1984 |
| Apple Watch | Wearable | 2015 |
| AirPods | Audio | 2016 |

These products have revolutionized their respective categories.
"""

# Table chunking
table_chunker = TableChunker(
    preserve_headers=True,
    max_rows_per_chunk=3,
    include_context=True,
    table_format="markdown"
)

chunks = table_chunker.chunk(text_with_table)

print("Table Chunking Results:\n")
print("=" * 80)

for i, chunk in enumerate(chunks, 1):
    is_table = chunk.metadata.get('is_table', False)
    
    print(f"\nChunk {i}:")
    print(f"  Type: {'Table' if is_table else 'Text'}")
    
    if is_table:
        rows = chunk.metadata.get('row_count', 'N/A')
        cols = chunk.metadata.get('column_count', 'N/A')
        print(f"  Rows: {rows}, Columns: {cols}")
    
    print(f"  Content: {chunk.text[:100]}...")

print("\n" + "=" * 80)

## 步骤 11:溯源追踪

追踪分块的来源,以实现数据血缘和调试。

### 为什么要追踪溯源?

- **数据血缘**:知道分块来自哪里
- **调试**:将问题追溯到源头
- **合规**:某些使用场景要求如此

In [ ]:
import sys
import os
import importlib

# 1. Ensure local package is in path
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# 2. Force unload modules to ensure clean reload
modules_to_unload = [
    'semantica.split.semantic_chunker', 
    'semantica.split.splitter', 
    'semantica.split.provenance_tracker',
    'semantica.split'
]
for module in modules_to_unload:
    if module in sys.modules:
        del sys.modules[module]

# 3. Import fresh modules
import semantica.split.semantic_chunker
import semantica.split.splitter
import semantica.split.provenance_tracker
from semantica.split import ProvenanceTracker, TextSplitter

# 4. Verify Chunk class has id field
from semantica.split.semantic_chunker import Chunk
print(f"Chunk class fields: {Chunk.__annotations__}")
if 'id' not in Chunk.__annotations__:
    print("WARNING: Chunk class still missing 'id' field. Kernel restart required.")

# Create chunks
splitter = TextSplitter(method="recursive", chunk_size=200, chunk_overlap=50)
chunks = splitter.split(text)

# Track provenance
tracker = ProvenanceTracker()

for chunk in chunks:
    tracker.track_chunk(
        chunk=chunk,
        source_document="apple_doc_001",
        source_path="data/apple.txt",
        timestamp="2024-01-01T00:00:00Z",
        method="recursive"
    )

print("Provenance Tracking Results:\n")
print("=" * 80)

# Get lineage for first chunk
if chunks:
    # Get provenance info using the chunk's ID
    chunk_id = getattr(chunks[0], 'id', None)
    print(f"Chunk ID: {chunk_id}")
    
    if chunk_id:
        prov_info = tracker.get_provenance(chunk_id)
        
        if prov_info:
            print(f"\nLineage for Chunk 1:")
            print(f"  Source Document: {prov_info.source_document}")
            print(f"  File Path: {prov_info.source_path}")
            print(f"  Method: {prov_info.metadata.get('method')}")
            print(f"  Timestamp: {prov_info.timestamp}")
    else:
        print("Error: Chunk ID not found. The Chunk class definition might still be cached.")
        print("Please click 'Kernel' -> 'Restart Kernel' in the menu and run all cells again.")

print("\n" + "=" * 80)

## 步骤 13:方法比较

让我们并排比较所有方法,帮助你选择合适的方法。

### 比较标准

- **分块数量**:创建的分块数
- **平均大小**:分块的平均大小
- **处理时间**:分块的速度

In [ ]:
import time

# Methods to compare
methods_to_compare = [
    ("recursive", {}),
    ("sentence", {}),
    ("paragraph", {}),
    ("token", {"tokenizer": "tiktoken"}),
]

print("Method Comparison:\n")
print("=" * 80)
print(f"{'Method':<15} {'Chunks':<10} {'Avg Size':<12} {'Time (ms)':<12}")
print("-" * 80)

for method, kwargs in methods_to_compare:
    try:
        start_time = time.time()
        
        splitter = TextSplitter(
            method=method,
            chunk_size=200,
            chunk_overlap=50,
            **kwargs
        )
        
        chunks = splitter.split(text)
        
        elapsed = (time.time() - start_time) * 1000
        avg_size = sum(len(c.text) for c in chunks) / len(chunks) if chunks else 0
        
        print(f"{method:<15} {len(chunks):<10} {avg_size:<12.0f} {elapsed:<12.2f}")
        
    except Exception as e:
        print(f"{method:<15} Error: {str(e)[:40]}")

print("=" * 80)

## 步骤 14:最佳实践

### 选择合适的方法

1. **通用文档**:使用 `recursive`,兼顾速度和简单性
2. **LLM 应用**:使用 `token` 以遵守上下文窗口
3. **语义搜索**:使用 `semantic_transformer` 保持主题连贯
4. **GraphRAG**:使用 `entity_aware` 或 `relation_aware`
5. **结构化文档**:对有格式的文档使用 `structural`
6. **大型文档**:使用 `hierarchical` 以支持多层级访问

### 分块大小指南

| 使用场景 | 推荐大小 | 重叠 |
|----------|------------------|----------|
| 语义搜索 | 512-1024 字符 | 20% |
| LLM 上下文 | 2000-4000 字符 | 10-20% |
| 实体抽取 | 500-1500 字符 | 15-25% |
| 问答 | 1000-2000 字符 | 20% |

### 重叠建议

- **10-15%**:处理快,冗余少
- **20-25%**:均衡(推荐)
- **30-40%**:最大程度保留上下文

## 总结

### 你学到了什么

在本 Notebook 中,你学会了如何:

- 使用 `TextSplitter` 调用多种方法
- 应用标准切分(recursive、token、sentence、paragraph)
- 使用语义分块以保持主题连贯
- 应用感知知识图谱的分块(entity-aware、relation-aware)
- 使用专用分块器(structural、hierarchical、sliding window、table)
- 追踪溯源
- 为你的使用场景选择合适的方法

### 关键要点

1. **方法选择很重要**:不同需求使用不同方法
2. **分块大小至关重要**:在上下文和处理之间取得平衡
3. **重叠有帮助**:20% 的重叠是不错的默认值
4. **追踪溯源**:对调试和合规很重要
5. **GraphRAG 使用感知知识图谱的分块**:为知识图谱使用 entity/relation-aware

### 下一步

**下一个 Notebook**:[12_Embedding_Generation.ipynb](./12_Embedding_Generation.ipynb)  
学习如何为你的分块生成嵌入!

**延伸阅读**:
- [Split 模块 API 参考](https://semantica.readthedocs.io/reference/split/)
- [高级分块策略](../advanced/11_Text_Chunking_Strategies.ipynb)
- [GraphRAG 管线](../use_cases/advanced_rag/01_GraphRAG_Complete.ipynb)

---

**有问题?**请查看我们的 [GitHub 仓库](https://github.com/semantica-agi/semantica)或[文档](https://semantica.readthedocs.io)。